In [0]:
%sql 
select * from 
"s3://sj-dbr-demo-proj/company_stocks.csv"

### Read file from S3 bucket


In [0]:
file_path = "s3://sj-dbr-demo-proj/company_stocks.csv"

df = spark.read \
    .format("csv") \
    .option("header",True) \
    .load(file_path)

In [0]:
df.write \
    .format('parquet') \
    .save("s3://sj-dbr-demo-proj/stock_parquet")

### Write this csv as Delta format table in DBT (Managed Table)

In [0]:
warehouse_location = "workspace.default.company_stocks"
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(warehouse_location)

#### Write same table as delta to S3 location(External Table)

In [0]:
df.show(2)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3://sj-dbr-demo-proj/bronze/")


### we cal list s3 files using dbutils 
 
display(dbutils.fs.ls ("s3://sj-dbr-demo-proj/bronze/"))

### How Delta tables work 

- now very important thing to notice here 
- As we have delta files stored in S3 , we can create table in DBR which can be either 
  A. Temp View 
  B. Parmamanent table stored in DBR DB ( Hive metastore)

- In option B - 
    - Only metadata of the table like 
    - Table name: workspace.default.company_stocks
    - Schema (column names and types)
    - Table location (s3://my-bucket/company_stocks_delta/)
    - Partitioning info (if any) 
    will be stored in DBT , if we perform any DML's delta log in S3 will be updated \

- Delta generates a new version of the table.
  - The _delta_log folder in S3 is updated:
  - JSON transaction log files track added/removed Parquet files.
  - Old Parquet files may remain until vacuumed (for time travel).
  -     The data files on S3 are modified according to the operation.
  ✅ So the S3 location always reflects the latest Delta state, even if the metastore only knows metadata.

In [0]:
%sql
create table workspace.default.company_stocks_cp 
using delta 
location "s3://sj-dbr-demo-proj/bronze/"




In [0]:
df = spark.read.table("workspace.default.company_stocks_cp")

In [0]:
df.show(5)


### Version , History etc for Time Travel **

In [0]:
%sql 
-- gives all versions for DML performed 
describe history  workspace.default.company_stocks_cp ;

-- Select psecific versions 
 select * from workspace.default.company_stocks_cp  version as of 3 

In [0]:
%sql 
update workspace.default.company_stocks_cp set Stock_Price = '150' where Company = 'apple'

### Delta & Parquet

- Imagine you are receiving a lot of data in plain CSV format from a client.  
- You read that data, applied some transformations, and stored it in S3/Azure via Databricks as **Parquet files**.  
- <span style="color: red;">**Benefits achieved so far with Parquet:**</span>
  - Columnar storage → efficient compression and faster reads
  - Partitioning → improves query performance for large datasets
- Now, suppose another user updates those Parquet files and stores them again.  
  - In this case, **the previous version of the data is lost**. There is no transaction log or versioning.  
- **Delta Lake** is built **on top of Parquet** to address this:  
  - Provides **ACID transactions** → safe concurrent reads and writes  
  - Supports **Time Travel** → query previous versions of data  
  - Adds **schema enforcement and evolution** → prevents invalid writes and handles schema changes  

### *** Save() Vs SaveAsTable() ***

#### Save()
- when we use this method to save table then all files are (delta files) written to underlaying storage ADLS / S3 etc.
- No tbale in DBR metastore is created. so we cant query using select * 
- we again have to read it using spark.read.format('delta') --> create temp view and then query it. 

 - E.g. df.write.format("delta").save("s3://bucket/staging/job1_output")

 - This is useful when we dont want evry table to be DBR table & have scenario of cross plattform sharing. 

 

### saveAstable()
- This actually registers table in DBR 
- can be queried in SQL context
- still have underlaying files in Storage 
- e.g. df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("path", "s3://my-bucket/my-table") \
  .saveAsTable("workspace.default.my_table")